In [6]:
!python -V

!which python

import sys, site, os
print("sys.executable:", sys.executable)
print("sys.prefix:", sys.prefix)
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX"))
print("site-packages:", site.getsitepackages())

Python 3.13.12
/opt/conda/bin/python
sys.executable: /opt/conda/envs/mlops-notebook/bin/python
sys.prefix: /opt/conda/envs/mlops-notebook
CONDA_PREFIX: /opt/conda
site-packages: ['/opt/conda/envs/mlops-notebook/lib/python3.13/site-packages']


In [7]:
import pandas as pd

In [8]:
import pickle

In [9]:
import seaborn as sns
import matplotlib.pyplot as plt

In [10]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [11]:
import mlflow

mlflow.set_tracking_uri("http://mlflow.ml.svc.cluster.local")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1772879221163, experiment_id='1', last_update_time=1772879221163, lifecycle_stage='active', name='nyc-taxi-experiment', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [12]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [13]:
!pwd

df_train = read_dataframe('./mlops-zoomcamp/01-intro/data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./mlops-zoomcamp/01-intro/data/green_tripdata_2021-02.parquet')

/home/jovyan


In [14]:
len(df_train), len(df_val)

(73908, 61921)

In [15]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [16]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [17]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [18]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.758715211164435

In [19]:
!mkdir -p "./mlops-zoomcamp/02-experiment-tracking/models"

with open('./mlops-zoomcamp/02-experiment-tracking/models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [20]:
with mlflow.start_run():

    mlflow.set_tag("developer", "cristian")

    mlflow.log_param("train-data-path", "./mlops-zoomcamp/01-intro/data/green_tripdata_2021-01.csv")
    mlflow.log_param("valid-data-path", "./mlops-zoomcamp/01-intro/data/green_tripdata_2021-02.csv")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    # push the model to mlflow
    mlflow.log_artifact(local_path="mlops-zoomcamp/02-experiment-tracking/models/lin_reg.bin", artifact_path="models_pickle")

🏃 View run mercurial-finch-247 at: http://mlflow.ml.svc.cluster.local/#/experiments/1/runs/09d557f50ef14145a63a7deae6ade94b
🧪 View experiment at: http://mlflow.ml.svc.cluster.local/#/experiments/1


In [21]:
import xgboost as xgb

In [22]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

/opt/conda/envs/mlops-notebook/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [23]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [24]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [19]:
# check if we have cuda in xgboost
print(xgb.__version__)
print(xgb.build_info())

3.2.0
{'BUILTIN_PREFETCH_PRESENT': True, 'CUDA_VERSION': [12, 9], 'DEBUG': False, 'GCC_VERSION': [14, 3, 0], 'GLIBC_VERSION': [2, 17], 'MM_PREFETCH_PRESENT': True, 'NCCL_VERSION': [2, 29, 3], 'THRUST_VERSION': [3, 2, 1], 'USE_CUDA': True, 'USE_DLOPEN_NCCL': False, 'USE_FEDERATED': False, 'USE_NCCL': True, 'USE_NVCOMP': False, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/opt/conda/envs/mlops-notebook/lib/libxgboost.so'}


In [20]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'device': 'cuda',
    'tree_method': 'hist',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:11.38289                          
[1]	validation-rmse:10.66259                          
[2]	validation-rmse:10.03902                          
[3]	validation-rmse:9.50107                           
[4]	validation-rmse:9.04008                           
[5]	validation-rmse:8.64538                           
[6]	validation-rmse:8.31059                           
[7]	validation-rmse:8.02496                           
[8]	validation-rmse:7.78526                           
[9]	validation-rmse:7.58213                           
[10]	validation-rmse:7.41070                          
[11]	validation-rmse:7.26636                          
[12]	validation-rmse:7.14527                          
[13]	validation-rmse:7.04300                          
[14]	validation-rmse:6.95694                          
[15]	validation-rmse:6.88605                          
[16]	validation-rmse:6.82590                          
[17]	validation-rmse:6.77361                          
[18]	valid

KeyboardInterrupt: 

In [120]:
mlflow.xgboost.autolog(disable=True)

In [ ]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.2308230301756129,
        'max_depth': 33,
        'min_child_weight': 1.0010990215608997,
        'objective': 'reg:squarederror',
        'reg_alpha': 0.19075036297361422,
        'reg_lambda': 0.3538062652238721,
        'device': 'cuda',
        'tree_method': 'hist',
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("./mlops-zoomcamp/02-experiment-tracking/models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)

    # log the preprocessor as an artifact, so we can load it later if we want to
    mlflow.log_artifact("mlops-zoomcamp/02-experiment-tracking/models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:10.39108
[1]	validation-rmse:9.12074
[2]	validation-rmse:8.25620
[3]	validation-rmse:7.67834
[4]	validation-rmse:7.29859
[5]	validation-rmse:7.04500
[6]	validation-rmse:6.87462
[7]	validation-rmse:6.76451
[8]	validation-rmse:6.69007
[9]	validation-rmse:6.63518
[10]	validation-rmse:6.59664
[11]	validation-rmse:6.56930
[12]	validation-rmse:6.54758
[13]	validation-rmse:6.53244
[14]	validation-rmse:6.52063
[15]	validation-rmse:6.51240
[16]	validation-rmse:6.50570
[17]	validation-rmse:6.49915
[18]	validation-rmse:6.49586
[19]	validation-rmse:6.49272
[20]	validation-rmse:6.49015
[21]	validation-rmse:6.48813
[22]	validation-rmse:6.48612
[23]	validation-rmse:6.48421
[24]	validation-rmse:6.48224
[25]	validation-rmse:6.48023
[26]	validation-rmse:6.47731
[27]	validation-rmse:6.47460
[28]	validation-rmse:6.47270
[29]	validation-rmse:6.47000
[30]	validation-rmse:6.46864
[31]	validation-rmse:6.46669
[32]	validation-rmse:6.46443
[33]	validation-rmse:6.46253
[34]	validation-rmse:6.

2026/03/07 14:36:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run awesome-gnu-284 at: https://mlflow.teom.org/#/experiments/1/runs/89b57f9a4aa0491bb76317c8bd5bf292
🧪 View experiment at: https://mlflow.teom.org/#/experiments/1


In [141]:
# same as before, but now with autologging!

mlflow.xgboost.autolog()

train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

best_params = {
    'learning_rate': 0.2308230301756129,
    'max_depth': 33,
    'min_child_weight': 1.0010990215608997,
    'objective': 'reg:squarederror',
    'reg_alpha': 0.19075036297361422,
    'reg_lambda': 0.3538062652238721,
    'device': 'cuda',
    'tree_method': 'hist',
    'seed': 42
}

#mlflow.log_params(best_params)

booster = xgb.train(
    params=best_params,
    dtrain=train,
    num_boost_round=1000,
    evals=[(valid, 'validation')],
    early_stopping_rounds=50
)

y_pred = booster.predict(valid)
rmse = root_mean_squared_error(y_val, y_pred)
mlflow.log_metric("rmse", rmse)

with open("./mlops-zoomcamp/02-experiment-tracking/models/preprocessor.b", "wb") as f_out:
    pickle.dump(dv, f_out)
#mlflow.log_artifact("mlops-zoomcamp/02-experiment-tracking/models/preprocessor.b", artifact_path="preprocessor")

mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:10.39108
[1]	validation-rmse:9.12074
[2]	validation-rmse:8.25620
[3]	validation-rmse:7.67834
[4]	validation-rmse:7.29859
[5]	validation-rmse:7.04500
[6]	validation-rmse:6.87462
[7]	validation-rmse:6.76451
[8]	validation-rmse:6.69007
[9]	validation-rmse:6.63518
[10]	validation-rmse:6.59664
[11]	validation-rmse:6.56930
[12]	validation-rmse:6.54758
[13]	validation-rmse:6.53244
[14]	validation-rmse:6.52063
[15]	validation-rmse:6.51240
[16]	validation-rmse:6.50570
[17]	validation-rmse:6.49915
[18]	validation-rmse:6.49586
[19]	validation-rmse:6.49272
[20]	validation-rmse:6.49015
[21]	validation-rmse:6.48813
[22]	validation-rmse:6.48612
[23]	validation-rmse:6.48421
[24]	validation-rmse:6.48224
[25]	validation-rmse:6.48023
[26]	validation-rmse:6.47731
[27]	validation-rmse:6.47460
[28]	validation-rmse:6.47270
[29]	validation-rmse:6.47000
[30]	validation-rmse:6.46864
[31]	validation-rmse:6.46669
[32]	validation-rmse:6.46443
[33]	validation-rmse:6.46253
[34]	validation-rmse:6.

2026/03/07 14:52:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 14:53:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./mlops-zoomcamp/01-intro/data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./mlops-zoomcamp/01-intro/data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("mlops-zoomcamp/02-experiment-tracking/models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        mlflow.sklearn.log_model(mlmodel, artifact_path="model")

2026/03/10 15:23:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


KeyboardInterrupt: 

In [28]:
# optimised for my own JupyterHub setup
import time
import mlflow
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
)
from sklearn.svm import LinearSVR
from sklearn.metrics import root_mean_squared_error

mlflow.sklearn.autolog(log_models=False)

models = [
    RandomForestRegressor(n_estimators=100, n_jobs=12, random_state=42),
    ExtraTreesRegressor(n_estimators=100, n_jobs=12, random_state=42),
    GradientBoostingRegressor(random_state=42),
    LinearSVR(random_state=42, max_iter=5000),
]

for mlmodel in models:
    model_name = mlmodel.__class__.__name__

    with mlflow.start_run(run_name=model_name):
        mlflow.set_tag("model_class", model_name)
        mlflow.log_param("train-data-path", "./mlops-zoomcamp/01-intro/data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./mlops-zoomcamp/01-intro/data/green_tripdata_2021-02.csv")
        mlflow.log_artifact(
            "mlops-zoomcamp/02-experiment-tracking/models/preprocessor.b",
            artifact_path="preprocessor",
        )

        t0 = time.time()
        print(f"Training {model_name}...")
        mlmodel.fit(X_train, y_train)
        fit_time = time.time() - t0
        print(f"Training {model_name} done in {fit_time:.2f}s")

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("fit_time_seconds", fit_time)
        mlflow.sklearn.log_model(mlmodel, artifact_path="model")

        print(f"{model_name:30s} RMSE={rmse:.4f}  fit_time={fit_time:.2f}s")

Training RandomForestRegressor...
Training RandomForestRegressor done in 51.16s


2026/03/10 21:13:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 21:13:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForestRegressor          RMSE=6.9054  fit_time=51.16s
🏃 View run RandomForestRegressor at: http://mlflow.ml.svc.cluster.local/#/experiments/1/runs/c9c6db67f5594eb7b0eabfccb47adf8b
🧪 View experiment at: http://mlflow.ml.svc.cluster.local/#/experiments/1
Training ExtraTreesRegressor...
Training ExtraTreesRegressor done in 173.51s


2026/03/10 21:41:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 21:41:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ExtraTreesRegressor            RMSE=6.9342  fit_time=173.51s
🏃 View run ExtraTreesRegressor at: http://mlflow.ml.svc.cluster.local/#/experiments/1/runs/e01c095b12e145cdb268760757e4f6cf
🧪 View experiment at: http://mlflow.ml.svc.cluster.local/#/experiments/1
Training GradientBoostingRegressor...
Training GradientBoostingRegressor done in 4.72s


...

GradientBoostingRegressor      RMSE=6.7423  fit_time=4.72s
🏃 View run GradientBoostingRegressor at: http://mlflow.ml.svc.cluster.local/#/experiments/1/runs/2bd0142bd24749fc9b408b75deb4cf9f
🧪 View experiment at: http://mlflow.ml.svc.cluster.local/#/experiments/1
Training LinearSVR...


/opt/conda/envs/mlops-notebook/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Training LinearSVR done in 9.04s


2026/03/10 22:09:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 22:09:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LinearSVR                      RMSE=24.6064  fit_time=9.04s
🏃 View run LinearSVR at: http://mlflow.ml.svc.cluster.local/#/experiments/1/runs/4478ef57c2c64d47be18112795aae15d
🧪 View experiment at: http://mlflow.ml.svc.cluster.local/#/experiments/1


In [ ]:
!curl  http://mlflow.ml.svc.cluster.local/api/2.0/mlflow-artifacts/artifacts/1/models/m-eff836d5a0cd414fad1e0b1f396084ae/artifacts/model.pkl

{"error_code": "INTERNAL_ERROR", "message": "The following failures occurred while downloading one or more artifacts from /mlflow/data/mlartifacts:\n##### File 1/models/m-eff836d5a0cd414fad1e0b1f396084ae/artifacts/model.pkl #####\n[Errno 2] No such file or directory: '/mlflow/data/mlartifacts/1/models/m-eff836d5a0cd414fad1e0b1f396084ae/artifacts/model.pkl'"}